# Search White House Speeches from 2021 to 2022 Based On Content

一个基于2021年至2022年白宫演讲的语义搜索示例。其中许多演讲是在GPT-3.5训练之后发表的。白宫（演讲与声明）2022年12月10日的数据集可在Kaggle上找到。本示例中，我们还将其上传至Google Drive。我们构建了一个系统，利用向量数据库和sentence-transformers库对这些演讲进行语义搜索。在此示例中，我们使用Milvus Lite在本地运行向量数据库。

首先，我们安装必要的库：

In [1]:
# ! pip install pymilvus sentence-transformers gdown milvus

In [2]:
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'

os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

os.environ['TORCH_HOME'] = CUSTOM_CACHE

## Download Dataset

Next, we download and extract our dataset

In [3]:
# import gdown
# url='https://drive.google.com/uc?id=10_sVL0UmEog7mczLedK5s1pnlDOz3Ukf'
# output='data/white_2021_2022.zip'
# gdown.download(url, output)
#
# import zipfile
# with zipfile.ZipFile(output, 'r') as zip_ref:
#     zip_ref.extractall("data/white_2021_2022")

## Clean the Data

该数据集未经预处理，因此我们需要先进行清理才能继续处理。第一步是使用 .dropna() 方法删除所有包含空值或 NaN 的行。接着，我们确保只选取字符数超过 50 个的演讲内容，以避免遗漏部分演讲。同时，我们去除演讲中的所有换行符和回车符。最后，将日期转换为普遍接受的 datetime 格式。

In [4]:
import pandas as pd
df=pd.read_csv('data/white_2021_2022/The white house speeches.csv')
df.head()

,Title,Date_time,Location,Speech
0,Remarks by President Biden Before Marine One D...,"OCTOBER 12, 2022",Not determined,NaN
1,Remarks by President Biden in a Virtual Recept...,"OCTOBER 11, 2022",Not determined,"6:47 P.M. EDT\r\n \r\nTHE PRESIDENT: Well, th..."
2,Remarks by President Biden at the Summit on Fi...,"OCTOBER 11, 2022",Eisenhower Executive Office Building,"2:56 P.M. EDT\r\n\r\nTHE PRESIDENT: Doctor, t..."
3,Remarks by Vice President Harris at a Democrat...,"OCTOBER 10, 2022","Princeton, New Jersey","THE VICE PRESIDENT: Good morning, everyone.\r..."
4,Remarks by Vice President Harris in a Conversa...,"OCTOBER 09, 2022","Austin, Texas",NaN


In [5]:
df=df.dropna()
cleaned_df=df.loc[(df["Speech"].str.len()>50)]
cleaned_df

,Title,Date_time,Location,Speech
1,Remarks by President Biden in a Virtual Recept...,"OCTOBER 11, 2022",Not determined,"6:47 P.M. EDT\r\n \r\nTHE PRESIDENT: Well, th..."
2,Remarks by President Biden at the Summit on Fi...,"OCTOBER 11, 2022",Eisenhower Executive Office Building,"2:56 P.M. EDT\r\n\r\nTHE PRESIDENT: Doctor, t..."
3,Remarks by Vice President Harris at a Democrat...,"OCTOBER 10, 2022","Princeton, New Jersey","THE VICE PRESIDENT: Good morning, everyone.\r..."
5,Remarks by Vice President Harris in a Keynote ...,"OCTOBER 09, 2022","Austin, Texas",5:44 P.M. CDT\r\n \r\nTHE VICE PRESIDENT: Go...
6,Remarks by President Biden on the Economy and ...,"OCTOBER 07, 2022","Hagerstown, Maryland","1:24 P.M. EDT\r\n\r\nTHE PRESIDENT: Please, h..."
...,...,...,...,...
1091,Remarks By Vice President Harris To State Depa...,"FEBRUARY 04, 2021",Harry S. Truman Building,"THE VICE PRESIDENT: Thank you, Secretary Blin..."
1095,Remarks by President Biden on the Fight to Con...,"JANUARY 26, 2021",Not determined,4:50 P.M. EST\r\n\r\n THE PRESIDENT: Than...
1096,Remarks by President Biden at Signing of an Ex...,"JANUARY 26, 2021",Not determined,2:06 P.M. EST \r\n THE PRESIDENT: Good af...
1097,REMARKS BY VICE PRESIDENT HARRIS AFTER RECEIVI...,"JANUARY 26, 2021","Bethesda, Maryland",3:53 P.M. EST\r\n\r\n THE VICE PRESIDENT: ...


In [6]:
cleaned_df['Speech']=cleaned_df['Speech'].str.replace("\r\n","")
cleaned_df.iloc[0]['Speech'][:300]

'6:47 P.M. EDT THE PRESIDENT:  Well, thank you very much.  And I thought I saw Fred Sears in that picture.  PARTICIPANT: You have. THE PRESIDENT:  And, by the way, you know, I owe — I owe Fred a debt of gratitude.  Years and years ago, he — he’s the reason why my first wife ended up marrying me.  We '

In [7]:
import datetime

# Convert the 'date' columns to datetime objects
cleaned_df['Date_time']=pd.to_datetime(cleaned_df['Date_time'],format='%B %d, %Y')

# Convert the datetime to Unix time format
cleaned_df['Date_time']=cleaned_df['Date_time'].apply(lambda x: int(x.timestamp()))

cleaned_df

,Title,Date_time,Location,Speech
1,Remarks by President Biden in a Virtual Recept...,1665446400,Not determined,"6:47 P.M. EDT THE PRESIDENT: Well, thank you ..."
2,Remarks by President Biden at the Summit on Fi...,1665446400,Eisenhower Executive Office Building,"2:56 P.M. EDTTHE PRESIDENT: Doctor, thank you..."
3,Remarks by Vice President Harris at a Democrat...,1665360000,"Princeton, New Jersey","THE VICE PRESIDENT: Good morning, everyone. A..."
5,Remarks by Vice President Harris in a Keynote ...,1665273600,"Austin, Texas",5:44 P.M. CDT THE VICE PRESIDENT: Good eveni...
6,Remarks by President Biden on the Economy and ...,1665100800,"Hagerstown, Maryland","1:24 P.M. EDTTHE PRESIDENT: Please, have a se..."
...,...,...,...,...
1091,Remarks By Vice President Harris To State Depa...,1612396800,Harry S. Truman Building,"THE VICE PRESIDENT: Thank you, Secretary Blin..."
1095,Remarks by President Biden on the Fight to Con...,1611619200,Not determined,4:50 P.M. EST THE PRESIDENT: Thank you fo...
1096,Remarks by President Biden at Signing of an Ex...,1611619200,Not determined,2:06 P.M. EST THE PRESIDENT: Good aftern...
1097,REMARKS BY VICE PRESIDENT HARRIS AFTER RECEIVI...,1611619200,"Bethesda, Maryland","3:53 P.M. EST THE VICE PRESIDENT: Well, s..."


## Establish a Vector Database and Schema

在完成所有数据清洗后，现在是时候设置我们的向量数据库 Milvus Lite 了。我们首先声明一些常量，然后启动服务器并建立连接。

In [8]:
COLLECTION_NAME='white_house_2021_2022'
DIMENSION=384
BATCH_SIZE=128
TOPK=3

In [9]:
from pymilvus import MilvusClient, DataType

mc=MilvusClient('../milvus_demo.db')

print(f"Type of server: {mc.get_server_version()}")

Type of server: milvus_lite-3.1.1


为了确保我们从零开始，我们会检查是否已存在与我们选择的名称相同的集合，并将其删除。

In [10]:
has=mc.has_collection(COLLECTION_NAME)
if has:
    mc.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: {COLLECTION_NAME}")


Successfully dropped collection: white_house_2021_2022


现在我们来建立数据模型。对于这个数据集，我们需要处理四个属性：演讲的标题、演讲日期、演讲地点以及演讲内容本身。我们希望对实际演讲的内容进行语义搜索，因此该模型将包含标题、日期、地点，以及演讲内容的向量嵌入表示。

对于每个VARCHAR数据类型（字符串格式），我们为其设定最大长度。在此情况下，这些最大长度均未被使用，但可作为粗略的上限估计值。

In [11]:
schema=mc.create_schema(enable_dynamic_fields=False)
schema.add_field(field_name='id', datatype=DataType.INT64,is_primary=True,auto_id=True)
schema.add_field(field_name='title', datatype=DataType.VARCHAR,max_length=500)
schema.add_field(field_name='date', datatype=DataType.VARCHAR,max_length=100)
schema.add_field(field_name='location', datatype=DataType.VARCHAR,max_length=200)
schema.add_field(field_name='embedding', datatype=DataType.FLOAT_VECTOR,dim=DIMENSION)

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 500}}, {'name': 'date', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}}, {'name': 'location', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 200}}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'enable_dynamic_field': False, 'enable_namespace': False}

在向量数据库服务器启动并建立好集合和模式后，插入向量前最后一步是创建向量索引。本例中，我们使用基于L2距离度量的IVF_FLAT索引，并设置128个聚类（nlist）。

In [12]:
index_params={
    "field_name":"embedding",
    "index_type":"IVF_FLAT",
    "metric_type":"L2",
    "params":{"nlist":128}
}
mc.create_collection(
    COLLECTION_NAME,
    consistency_level="Eventually",
    params=index_params,
    schema=schema
)

## Get Vector Embeddings and Populate the Database

我们使用 SentenceTransformer 库来获取演讲的向量嵌入，并将新生成的向量嵌入填充到 Milvus 实例中。在此示例中，我们使用 MiniLM L6 v2 模型生成向量嵌入。

In [13]:
from sentence_transformers import SentenceTransformer

transformer=SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


我们创建一个名为 embed_insert 的函数，用于获取一组演讲的嵌入向量，然后将该批次数据插入到我们的 Milvus 实例中。

In [21]:
def embed_insert(data:list):
    embeddings=transformer.encode(data[3])
    ins = [
            {
                "title": data[0][i],
                "date": data[1][i],
                "location": data[2][i],
                "embedding": embeddings[i],       # numpy 行向量,pymilvus 可直接接受
            }
            for i in range(len(data[0]))
        ]
    mc.insert(
        collection_name=COLLECTION_NAME,
        data=ins
    )

在编写好我们的辅助函数后，我们就可以将文本嵌入并插入了。首先，我们将pandas数据框转换为合适的格式——即列表的列表，以便进行插入。在这个示例中，我们需要一个包含四个子列表的列表。每个子列表分别对应标题、日期、地点和演讲内容。我们将这些列表批量处理，并对每一个子列表调用之前编写的embed_insert函数。最后，在所有数据插入完成后，我们刷新集合以确保所有内容都被正确索引。

In [22]:
date_batch=[[],[],[],[]]

for index,row in cleaned_df.iterrows():
    date_batch[0].append(row["Title"])
    date_batch[1].append(str(row["Date_time"]))
    date_batch[2].append(row["Location"])
    date_batch[3].append(row["Speech"])

    if len(date_batch[0])==BATCH_SIZE:
        embed_insert(date_batch)
        date_batch=[[],[],[],[]]

# Embed and insert the remainder
if len(date_batch[0])>0:
    embed_insert(date_batch)


## Run a Semantic Search

数据库填充完成后，现在可以基于内容搜索所有演讲。在此示例中，我们搜索一篇由总统在国家可再生能源实验室（NREL）发表关于可再生能源的演讲，以及一篇由加拿大副总统和总理共同发表的演讲。我们获取这些描述的嵌入向量，然后在向量数据库中搜索与之最接近的三篇演讲。

我们预计，第一段描述的结果中会包含题为“拜登总统视察国家可再生能源实验室时的讲话”，第二段描述的结果中则会包含题为“加拿大副总统哈里斯与总理特鲁多在双边会议前的讲话”。

In [25]:
import time
search_terms=["The President speaks about the impact of renewable energy at the National Renewable Energy Lab.",
              "The Vice President and the Prime Minister of Canada both speak."]

# Search the database baesd on input text
def embed_search(data):
    embeds=transformer.encode(data)
    return [x for x in embeds]

search_data=embed_search(search_terms)

start_time=time.time()
res=mc.search(
    collection_name=COLLECTION_NAME,
    data=search_data,
    anns_field="embedding",
    search_params={
        "metric_type":"L2",
        "params":{"nprobe":10}
    },
    limit=TOPK,
    output_fields=['title','date','location']
)
end_time=time.time()

for hits_i, hits in enumerate(res):
    print("Title: ",search_terms[hits_i])
    print("Search Tiem: ",end_time-start_time)
    print("Results: ")
    for hit in hits:
        print(hit.entity.get("title"),"----",hit.distance)
    print()

Title:  The President speaks about the impact of renewable energy at the National Renewable Energy Lab.
Search Tiem:  0.10500144958496094
Results: 
Remarks by President Biden During a Tour of the National Renewable Energy Laboratory ---- 1.0144073963165283
Press Gaggle by Vice President Harris Aboard Air Force Two Before Departure ---- 1.0430814027786255
Remarks by President Biden at the Virtual Leaders Summit on Climate Opening Session ---- 1.0471301078796387

Title:  The Vice President and the Prime Minister of Canada both speak.
Search Tiem:  0.10500144958496094
Results: 
REMARKS BY VICE PRESIDENT HARRIS AND PRIME MINISTER TRUDEAU OF CANADA BEFORE BILATERAL MEETING ---- 0.8196960091590881
Remarks by Vice President Harris After Meeting to Discuss the Importance of Passing the Build Back Better Agenda ---- 0.9929053783416748
Remarks by President Biden and Prime Minister Boris Johnson of the United Kingdom Before Bilateral Meeting ---- 1.0264472961425781



Clean up the server.

In [26]:
mc.close()